In [15]:
import pandas as pd
import digitalhub as dh
from digitalhub import get_model

from data_preparation import *
from modelling import *

# Project creation

In [3]:
project = dh.get_or_create_project("fair_hiring_tabular")

# Data preparation stage

In [4]:
URL = "https://raw.githubusercontent.com/AlbanaCelepija/enhanced_mlops/refs/heads/main/framework/library/use_cases/tabular/src/local_platform/artifacts/data/recruitmentdataset-2022.csv"
di = project.new_dataitem(name="recruitmentdataset.csv",
                          kind="table",
                          path=URL)
di

{'kind': 'table', 'metadata': {'project': 'fair_hiring_tabular', 'name': 'recruitmentdataset.csv', 'version': 'deep-hawk', 'created': '2026-03-05T11:23:11.61Z', 'updated': '2026-03-05T11:23:11.61Z', 'created_by': 'acelepija@fbk.eu', 'updated_by': 'acelepija@fbk.eu', 'embedded': False}, 'spec': {'path': 'https://raw.githubusercontent.com/AlbanaCelepija/enhanced_mlops/refs/heads/main/framework/library/use_cases/tabular/src/local_platform/artifacts/data/recruitmentdataset-2022.csv'}, 'status': {'state': 'CREATED', 'files': []}, 'user': 'acelepija@fbk.eu', 'project': 'fair_hiring_tabular', 'name': 'recruitmentdataset.csv', 'id': '2a38f7792ac34d578efc6323024cde2d', 'key': 'store://fair_hiring_tabular/dataitem/table/recruitmentdataset.csv:2a38f7792ac34d578efc6323024cde2d'}

In [5]:
training_di = project.get_dataitem('recruitmentdataset.csv')
training_df = training_di.as_df()
training_df

,Id,gender,age,nationality,sport,ind-university_grade,ind-debateclub,ind-programming_exp,ind-international_exp,ind-entrepeneur_exp,ind-languages,ind-exact_study,ind-degree,company,decision
0,x8011e,female,24,German,Swimming,70,False,False,False,False,1,True,phd,A,True
1,x6077a,male,26,German,Golf,67,False,True,False,False,2,True,bachelor,A,False
2,x6006e,female,23,Dutch,Running,67,False,True,True,False,0,True,master,A,False
3,x2173b,male,24,Dutch,Cricket,70,False,True,False,False,1,True,master,A,True
4,x6241a,female,26,German,Golf,59,False,False,False,False,1,False,master,A,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,x7640e,female,28,Dutch,Running,63,False,False,False,False,0,False,master,D,False
3996,x3310f,female,27,Dutch,Tennis,62,False,False,False,True,2,True,bachelor,D,False
3997,x1202g,male,24,Belgian,Rugby,60,True,False,False,True,2,False,bachelor,D,False
3998,x1263d,female,22,Dutch,Tennis,66,False,True,False,False,1,True,bachelor,D,False


In [6]:
data_preparation_fn = project.new_function(name="data_preprocessing",
                                   kind="python",
                                   python_version="PYTHON3_10",
                                   code_src="mini_data_preparation.py",
                                   handler="load_data",
                                   requirements=["scikit-learn"])
data_preparation_fn = data_preparation_fn.run("job",wait=True)
dataset = data_preparation_fn.output("dataset")

2026-03-05 11:23:12,789 - INFO - Waiting for run c61fc45021ac4364bc9aad7147a440c9 to finish...
INFO:digitalhub-core:Waiting for run c61fc45021ac4364bc9aad7147a440c9 to finish...
2026-03-05 11:23:17,806 - INFO - Waiting for run c61fc45021ac4364bc9aad7147a440c9 to finish...
INFO:digitalhub-core:Waiting for run c61fc45021ac4364bc9aad7147a440c9 to finish...
2026-03-05 11:23:22,821 - INFO - Waiting for run c61fc45021ac4364bc9aad7147a440c9 to finish...
INFO:digitalhub-core:Waiting for run c61fc45021ac4364bc9aad7147a440c9 to finish...
2026-03-05 11:23:27,839 - INFO - Waiting for run c61fc45021ac4364bc9aad7147a440c9 to finish...
INFO:digitalhub-core:Waiting for run c61fc45021ac4364bc9aad7147a440c9 to finish...
2026-03-05 11:23:32,857 - INFO - Waiting for run c61fc45021ac4364bc9aad7147a440c9 to finish...
INFO:digitalhub-core:Waiting for run c61fc45021ac4364bc9aad7147a440c9 to finish...
2026-03-05 11:23:37,881 - INFO - Waiting for run c61fc45021ac4364bc9aad7147a440c9 to finish...
INFO:digitalhub

In [7]:
data_preparation_fn.output("dataset").as_df().head()

,sport_Chess,sport_Cricket,sport_Football,sport_Golf,sport_Rugby,sport_Running,sport_Swimming,sport_Tennis,ind-degree_bachelor,ind-degree_master,...,age,nationality,ind-university_grade,ind-debateclub,ind-programming_exp,ind-international_exp,ind-entrepeneur_exp,ind-languages,ind-exact_study,decision
0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,24,German,70,0,0,0,0,1,1,1
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,26,German,67,0,1,0,0,2,1,0
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,23,Dutch,67,0,1,1,0,0,1,0
3,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,24,Dutch,70,0,1,0,0,1,1,1
4,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,...,26,German,59,0,0,0,0,1,0,1


## Split train-validate-test set

In [ ]:
config = {"test_size": 0.2, "valid_size": 0.25, "random_state": 42} # 0.25 x 0.8 = 0.2

split_fn = project.new_function(
    name="split-train-valid-test",
    kind="python",
    python_version="PYTHON3_10",
    code_src="mini_data_preparation.py",
    handler="split_train_valid_test_data",
    requirements=["scikit-learn", "numpy<2"],
)
split_run = split_fn.run(action="job", inputs={"data": dataset.key}, parameters=config, wait=True)


2026-03-05 11:29:22,125 - INFO - Waiting for run e68e2461bbf64290bbf28a94d2d71ff0 to finish...
INFO:digitalhub-core:Waiting for run e68e2461bbf64290bbf28a94d2d71ff0 to finish...
2026-03-05 11:29:27,138 - INFO - Waiting for run e68e2461bbf64290bbf28a94d2d71ff0 to finish...
INFO:digitalhub-core:Waiting for run e68e2461bbf64290bbf28a94d2d71ff0 to finish...
2026-03-05 11:29:32,158 - INFO - Waiting for run e68e2461bbf64290bbf28a94d2d71ff0 to finish...
INFO:digitalhub-core:Waiting for run e68e2461bbf64290bbf28a94d2d71ff0 to finish...
2026-03-05 11:29:37,179 - INFO - Waiting for run e68e2461bbf64290bbf28a94d2d71ff0 to finish...
INFO:digitalhub-core:Waiting for run e68e2461bbf64290bbf28a94d2d71ff0 to finish...
2026-03-05 11:29:42,196 - INFO - Waiting for run e68e2461bbf64290bbf28a94d2d71ff0 to finish...
INFO:digitalhub-core:Waiting for run e68e2461bbf64290bbf28a94d2d71ff0 to finish...
2026-03-05 11:29:47,227 - INFO - Waiting for run e68e2461bbf64290bbf28a94d2d71ff0 to finish...
INFO:digitalhub

# Modelling stage

In [ ]:
train_fn = project.new_function(
    name="train-classifier",
    kind="python",
    python_version="PYTHON3_10",
    code_src="mini_modelling.py",
    handler="train_model",
    requirements=["scikit-learn", "numpy<2"],
)
train_ds = project.get_dataitem('training_set_X')
train_run = train_fn.run(action="job", inputs={"data": train_ds.key}, wait=True)

In [ ]:
model = train_run.output("model")
model_obj = get_model(model.key)

## Evaluate overall performance metrics

In [ ]:
config={"sensitive_features": ["nationality", "gender"]}
valid_ds = project.get_dataitem('validation_set_X')

model_evaluation_accuracy_overall_fn = project.new_function(
    name="model-evaluation-accuracy-overall",
    kind="python",
    python_version="PYTHON3_10",
    code_src="mini_modelling.py",
    handler="model_evaluation_accuracy_overall",
    requirements=["scikit-learn", "numpy<2"],
)
model_evaluation_accuracy_overall_run = model_evaluation_accuracy_overall_fn.run(
    action="job",
    inputs={"model": model.key, "data_valid": valid_ds.key}, 
    parameters=config, 
    wait=True
)

## Evaluate performance metrics for each demographic group

In [ ]:
config={"sensitive_features": ["nationality", "gender"]}
valid_ds = project.get_dataitem('validation_set_X')

model_evaluation_accuracy_demographic_groups_fn = project.new_function(
    name="model-evaluation-accuracy-demographic-groups",
    kind="python",
    python_version="PYTHON3_10",
    code_src="mini_modelling.py",
    handler="model_evaluation_accuracy_demographic_groups",
    requirements=["scikit-learn", "numpy<2"],
)
model_evaluation_accuracy_demographic_groups_run = model_evaluation_accuracy_demographic_groups_fn.run(
    action="job",
    inputs={"model": model.key, "data_valid": valid_ds.key}, 
    parameters=config, 
    wait=True
)

# Operationalisation stage

In [ ]:
serve_func = project.new_function(
    name="serve-classifier",
    kind="sklearnserve",
    path=model.key,
)
serve_run = serve_func.run("serve", wait=True)
serve_run

# Inference 

In [ ]:
test_df = data_preparation_fn.output("dataset").as_df().head()
test_df = test_df.drop(columns=["decision", "Id", "gender", "nationality"])
test_df


In [ ]:
data = test_df.values.tolist()
json_payload = {"inputs": [{"name": "input-0", "shape": [5, 23], "datatype": "FP32", "data": data}]}
json_payload

In [ ]:
result = serve_run.refresh().invoke(json=json_payload).json()
print("Prediction result:")
print(result)

# Workflow orchestration

## Baseline Requirements Dimension 

In [ ]:
workflow_baseline = project.new_workflow(
    name="baseline-pipeline",
    kind="hera",
    code_src="pipeline.py",
    handler="baseline_pipeline",
)

In [ ]:
wf_build = workflow_baseline.run("build", wait=True)
wf_run = workflow_baseline.run("pipeline", wait=True)

## Fairness Requirements Dimension

In [ ]:
workflow_fairness = project.new_workflow(
    name="baseline-pipeline",
    kind="hera",
    code_src="pipeline.py",
    handler="fairness_pipeline",
)

In [ ]:
wf_build = workflow_fairness.run("build", wait=True)
wf_run = workflow_fairness.run("pipeline", wait=True)